In [3]:
import pandas as pd
import datetime


df = pd.read_csv("synthetic_commodity_2026.csv")

df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

price = df[[
    "Date",
    "WHEAT",
    "SUGAR",
    "COFFEE",
    "COTTON",
    "SOYBEANS"
]]

price.to_csv(
    "clean_prices.csv",
    index=False
)

In [4]:
bad_dates = df[df["Date"].isna()]

print(bad_dates)

Empty DataFrame
Columns: [Date, COFFEE, SUGAR, SOYBEANS, WHEAT, COTTON]
Index: []


In [5]:
import pandas as pd

price = pd.read_csv(
    "clean_prices.csv"
)

price_long = price.melt(
    id_vars="Date",
    var_name="commodity",
    value_name="price"
)

price_long.dropna(
    inplace=True
)

price_long.to_csv(
    "price_long.csv",
    index=False
)

print(price_long.head())

         Date commodity   price
0  2000-02-01     WHEAT  258.00
1  2000-03-01     WHEAT  247.75
2  2000-05-01     WHEAT  251.00
3  2000-06-01     WHEAT  270.50
4  2000-08-01     WHEAT  245.00


In [6]:
import pandas as pd

df = pd.read_csv(
    "price_long.csv"
)

df["Date"] = pd.to_datetime(
    df["Date"]
)

df = df.sort_values(
    ["commodity","Date"]
)

df["price_change"] = (
    df.groupby("commodity")
    ["price"]
    .diff()
)

df["price_pct_change"] = (
    df.groupby("commodity")
    ["price"]
    .pct_change()
)

df["MA7"] = (
    df.groupby("commodity")
    ["price"]
    .transform(
        lambda x:x.rolling(7).mean()
    )
)

df["MA30"] = (
    df.groupby("commodity")
    ["price"]
    .transform(
        lambda x:x.rolling(30).mean()
    )
)

df.to_csv(
    "price_features.csv",
    index=False
)

In [7]:
import pandas as pd

price = pd.read_csv(
    "price_features.csv"
)

price["Date"] = pd.to_datetime(
    price["Date"]
)

price["year"] = price["Date"].dt.year

price["month"] = price["Date"].dt.month

monthly_price = (

    price.groupby(
        [
            "commodity",
            "year",
            "month"
        ]
    )
    .agg({

        "price":"mean",

        "price_change":"mean",

        "price_pct_change":"mean",

        "MA7":"mean",

        "MA30":"mean"

    })

    .reset_index()
)

monthly_price.to_csv(
    "monthly_price.csv",
    index=False
)

In [8]:
mapping = {

    "wheat":"WHEAT",

    "sugar":"SUGAR",

    "coffee":"COFFEE",

    "cotton":"COTTON",

    "soybean":"SOYBEANS"
}

In [9]:
import pandas as pd

news = pd.read_csv(
    "commodity_news_dataset_90.csv"
)

news["commodity"] = (
    news["commodity"]
    .map(mapping)
)

news.dropna(
    inplace=True
)

news.to_csv(
    "mapped_news.csv",
    index=False
)

In [10]:
import pandas as pd


price = pd.read_csv(
    "monthly_price.csv"
)

news = pd.read_csv(
    "mapped_news.csv"
)


dataset = pd.merge(
    price,
    news,
    on=[
        "commodity",
        "year",
        "month"
    ],
    how="left"
)


# fill old years where no news exists

news_cols = [
"total_news",
"average_sentiment",
"positive_news",
"negative_news",
"neutral_news",
"news_growth"
]


for col in news_cols:
    dataset[col] = dataset[col].fillna(0)



dataset.to_csv(
    "final_ml_dataset.csv",
    index=False
)


print(dataset.shape)

(1620, 14)


In [11]:
dataset["total_news"] = dataset["total_news"].fillna(0)

dataset["average_sentiment"] = dataset["average_sentiment"].fillna(0)

dataset["positive_news"] = dataset["positive_news"].fillna(0)

dataset["negative_news"] = dataset["negative_news"].fillna(0)

dataset["neutral_news"] = dataset["neutral_news"].fillna(0)

dataset["news_growth"] = dataset["news_growth"].fillna(0)

In [12]:
df = pd.read_csv(
    "final_ml_dataset.csv"
)


df = df.sort_values(
[
"commodity",
"year",
"month"
]
)


df["next_price"] = (
    df.groupby("commodity")
    ["price"]
    .shift(-1)
)


df.dropna(inplace=True)


df.to_csv(
"training_dataset.csv",
index=False
)

In [13]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import joblib


# ==========================
# Load dataset
# ==========================

df = pd.read_csv(
    "training_dataset.csv"
)


# sort time properly

df = df.sort_values(
    [
        "commodity",
        "year",
        "month"
    ]
)


# ==========================
# Features
# ==========================

features = [

    # news features
    "total_news",
    "average_sentiment",
    "positive_news",
    "negative_news",
    "neutral_news",
    "news_growth",

    # price features
    "price",
    "price_change",
    "price_pct_change",
    "MA7",
    "MA30"

]


X = df[features]

y = df["next_price"]



# ==========================
# Handle missing values
# ==========================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)



# ==========================
# Time based split
# ==========================

train = df["year"] < 2025

test = df["year"] >= 2025


X_train = X[train]

y_train = y[train]


X_test = X[test]

y_test = y[test]



print(
    "Train:",
    X_train.shape
)

print(
    "Test:",
    X_test.shape
)



# ==========================
# XGBoost Model
# ==========================

model = XGBRegressor(

    n_estimators=800,

    learning_rate=0.03,

    max_depth=5,

    min_child_weight=3,

    subsample=0.8,

    colsample_bytree=0.8,

    objective="reg:squarederror",

    random_state=42

)



# ==========================
# Train
# ==========================

model.fit(

    X_train,

    y_train

)



# ==========================
# Prediction
# ==========================

pred = model.predict(
    X_test
)



# ==========================
# Evaluation
# ==========================

r2 = r2_score(
    y_test,
    pred
)


mae = mean_absolute_error(
    y_test,
    pred
)


rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)



print(
    "R2 Score:",
    r2
)

print(
    "MAE:",
    mae
)

print(
    "RMSE:",
    rmse
)



# ==========================
# Feature importance
# ==========================

importance = pd.DataFrame({

    "feature":features,

    "importance":
    model.feature_importances_

})


importance = importance.sort_values(
    "importance",
    ascending=False
)


print(
    importance
)



# ==========================
# Save model
# ==========================

joblib.dump(
    model,
    "commodity_xgboost.pkl"
)


print(
    "Model Saved"
)

Train: (1485, 11)
Test: (115, 11)
R2 Score: 0.9961606982255903
MAE: 17.91724498271846
RMSE: 39.9061889113056
              feature  importance
6               price    0.502166
9                 MA7    0.357803
10               MA30    0.133197
8    price_pct_change    0.003641
7        price_change    0.003193
0          total_news    0.000000
1   average_sentiment    0.000000
2       positive_news    0.000000
5         news_growth    0.000000
3       negative_news    0.000000
4        neutral_news    0.000000
Model Saved


In [14]:
print(model.score(X_train,y_train))

0.9988802499365963


In [15]:
print(model.score(X_test,y_test))

0.9961606982255903


In [16]:
print(df.shape)
print(df.head())

(1600, 15)
  commodity  year  month      price  price_change  price_pct_change  \
0    COFFEE  2000      4  98.981250     -1.487500         -0.014628   
1    COFFEE  2000      5  96.710000     -0.240000         -0.002122   
2    COFFEE  2000      6  92.512500     -0.081250         -0.000591   
3    COFFEE  2000      7  85.318750     -0.181250         -0.001182   
4    COFFEE  2000      8  83.672222     -1.005556         -0.011555   

          MA7        MA30  total_news  average_sentiment  positive_news  \
0  102.686607  108.383750         0.0                0.0            0.0   
1   97.196429  103.954333         0.0                0.0            0.0   
2   93.916964   99.197083         0.0                0.0            0.0   
3   87.210714   94.432708         0.0                0.0            0.0   
4   85.430952   90.486296         0.0                0.0            0.0   

   negative_news  neutral_news  news_growth  next_price  
0            0.0           0.0          0.0   96.7100

In [17]:
print(df.columns)



Index(['commodity', 'year', 'month', 'price', 'price_change',
       'price_pct_change', 'MA7', 'MA30', 'total_news', 'average_sentiment',
       'positive_news', 'negative_news', 'neutral_news', 'news_growth',
       'next_price'],
      dtype='str')


In [18]:
print(len(df))

1600


In [19]:
df = pd.read_csv(
    "training_dataset.csv"
)


last = df.tail(1)


X_new = last[
[
"total_news",
"average_sentiment",
"positive_news",
"negative_news",
"neutral_news",
"news_growth",
"price",
"price_change",
"price_pct_change",
"MA7",
"MA30"
]
]


pred = model.predict(
    X_new
)


print("Actual next price:", last["next_price"].values[0])

print("Predicted:", pred[0])

Actual next price: 633.3312791331035
Predicted: 624.1721


In [20]:
import pandas as pd
import joblib


# load trained model
model = joblib.load(
    "commodity_xgboost.pkl"
)


# realistic future input

new_market_data = pd.DataFrame({

    # news information
    "total_news":[150],

    "average_sentiment":[-0.35],

    "positive_news":[20],

    "negative_news":[90],

    "neutral_news":[40],

    "news_growth":[120],


    # current price information
    "price":[250.00],

    "price_change":[8.5],

    "price_pct_change":[0.034],


    # trend information
    "MA7":[245.50],

    "MA30":[235.20]

})


prediction = model.predict(
    new_market_data
)


print(
    "Predicted next month wheat price:",
    prediction[0]
)

Predicted next month wheat price: 244.69736
